# Explorateur de Mapping Complexe - Neural Form Finding
Ce notebook permet de visualiser l'effet de la transformation polynomiale complexe utilisée pour l'initial mapping. 
L'objectif est de trouver le bon compromis entre la déformation géométrique et la validité topologique (éviter le criss-cross).

In [34]:
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
%matplotlib inline
import numpy as np

In [35]:
def complex_map(u, v, domain_restriction, params):
    # 1. Pré-scaling (Restriction du domaine source)
    z = (u + 1j * v) * domain_restriction
    
    # Extraction des paramètres (Format: [tx, ty, theta, scale, c1, c2, ...])
    tx, ty = params[0], params[1]
    theta = params[2]
    s_val = params[3]
    c_vals = params[4:]
    
    # 2. Application du polynôme (Symétrie 4-fold)
    w = z
    for k, ck in enumerate(c_vals):
        w = w + ck * (z ** (5 * (k + 1) + 1))
    
    # 3. Post-transformation (Echelle, Rotation, Translation)
    w = s_val * w * np.exp(1j * theta)
    return np.real(w) + tx, np.imag(w) + ty

In [36]:
def visualize_mapping(domain_restriction, c1, c2, scale, rotation):
    # Création d'une grille régulière
    res = 20
    u_vals = np.linspace(-1, 1, res)
    v_vals = np.linspace(-1, 1, res)
    U, V = np.meshgrid(u_vals, v_vals)
    
    params = np.array([0.0, 0.0, np.radians(rotation), scale, c1, c2])
    
    X, Y = complex_map(U, V, domain_restriction, params)
    
    plt.figure(figsize=(10, 10))
    
    # Plot de la grille transformée
    for i in range(res):
        plt.plot(X[i, :], Y[i, :], 'b-', alpha=0.6)
        plt.plot(X[:, i], Y[:, i], 'b-', alpha=0.6)
        
    # Détection simplifiée des retournements (Jacobien)
    # On regarde l'aire des petits carreaux
    for i in range(res-1):
        for j in range(res-1):
            # Vecteurs de bords
            v1x, v1y = X[i+1, j] - X[i, j], Y[i+1, j] - Y[i, j]
            v2x, v2y = X[i, j+1] - X[i, j], Y[i, j+1] - Y[i, j]
            area = v1x * v2y - v1y * v2x
            if area < 0:
                plt.fill([X[i,j], X[i+1,j], X[i+1,j+1], X[i,j+1]], 
                         [Y[i,j], Y[i+1,j], Y[i+1,j+1], Y[i,j+1]], 'r', alpha=0.3)
    
    plt.gca().set_aspect('equal')
    plt.title(f"Mapping avec Restriction: {domain_restriction}")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()

In [ ]:
interact(visualize_mapping, 
         domain_restriction=FloatSlider(value=0.8, min=0.1, max=1.0, step=0.05),
         c1=FloatSlider(value=0.1, min=-0.5, max=0.5, step=0.01),
         c2=FloatSlider(value=0.0, min=-0.5, max=0.5, step=0.01),
         scale=FloatSlider(value=1.0, min=0.5, max=1.5),
         rotation=IntSlider(value=0, min=0, max=360))

interactive(children=(FloatSlider(value=0.8, description='domain_restriction', max=1.0, min=0.1, step=0.05), F…

<function __main__.visualize_mapping(domain_restriction, c1, c2, scale, rotation)>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def map_from_derivative_roots_np(z, roots):
    """
    Construit et évalue un mapping complexe à partir des racines de sa dérivée.
    Version pure NumPy, compatible avec le broadcasting sur des grilles 2D.
    """
    # 1. Construction des coefficients de f'(z) = prod(1 - z/r_i)
    coeffs = np.array([1.0 + 0j])
    for r in roots:
        shifted = np.pad(coeffs, (1, 0))
        coeffs = np.pad(coeffs, (0, 1)) - (1.0 / r) * shifted
        
    # 2. Intégration analytique exacte : a_k z^k -> (a_k / (k+1)) z^{k+1}
    k = np.arange(1, len(coeffs) + 1)
    integrated_coeffs = coeffs / k
    # Ajout du terme constant f(0) = 0
    integrated_coeffs = np.pad(integrated_coeffs, (1, 0)) 
    
    # 3. Évaluation du polynôme via np.polyval (nécessite l'ordre décroissant des puissances)
    return np.polyval(integrated_coeffs[::-1], z)


def visualize_roots_mapping_np(r1_mag, r1_phase, r2_mag, r2_phase, r3_mag, r3_phase):
    res = 20
    u_vals = np.linspace(-1, 1, res)
    v_vals = np.linspace(-1, 1, res)
    U, V = np.meshgrid(u_vals, v_vals)
    
    # Grille complexe 2D
    Z = U + 1j * V
    
    # Construction des racines à partir des coordonnées polaires
    r1 = r1_mag * np.exp(1j * np.radians(r1_phase))
    r2 = r2_mag * np.exp(1j * np.radians(r2_phase))
    r3 = r3_mag * np.exp(1j * np.radians(r3_phase))
    roots = np.array([r1, r2, r3])
    
    # Évaluation vectorisée native sur toute la grille Z
    W = map_from_derivative_roots_np(Z, roots)
    X, Y = np.real(W), np.imag(W)
    
    plt.figure(figsize=(10, 10))
    
    # Tracé du maillage déformé
    for i in range(res):
        plt.plot(X[i, :], Y[i, :], 'b-', alpha=0.6) # Lignes horizontales
        plt.plot(X[:, i], Y[:, i], 'b-', alpha=0.6) # Lignes verticales
        
    # Affichage des racines
    roots_x, roots_y = np.real(roots), np.imag(roots)
    plt.scatter(roots_x, roots_y, color='red', marker='x', s=100, label="Racines de f'(z)")
    
    plt.gca().set_aspect('equal')
    plt.title(f"Mapping Numpy paramétré par {len(roots)} singularités")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()

interact(visualize_roots_mapping_np, 
         r1_mag=FloatSlider(value=1.5, min=1.01, max=5.0, step=0.1, description='|R1| (Garde > 1)'),
         r1_phase=FloatSlider(value=45, min=0, max=360, step=5, description='Angle R1'),
         r2_mag=FloatSlider(value=2.0, min=1.01, max=5.0, step=0.1, description='|R2|'),
         r2_phase=FloatSlider(value=180, min=0, max=360, step=5, description='Angle R2'),
         r3_mag=FloatSlider(value=1.8, min=1.01, max=5.0, step=0.1, description='|R3|'),
         r3_phase=FloatSlider(value=270, min=0, max=360, step=5, description='Angle R3'))

interactive(children=(FloatSlider(value=1.5, description='|R1| (Garde > 1)', max=5.0, min=1.01), FloatSlider(v…

<function __main__.visualize_roots_mapping_np(r1_mag, r1_phase, r2_mag, r2_phase, r3_mag, r3_phase)>